In [134]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [305]:
df = pd.read_csv('insurance_premium_dataset.csv')

In [306]:
df.sample(5)


,age,weight,height,income_dkk,smoker,city,occupation,insurance_premium_category
424,50,96,194,301253,No,Kolding,retired,Low
867,54,71,185,566639,No,Aalborg,unemployed,Medium
16,56,95,187,564806,No,Silkeborg,freelancer,Medium
352,56,94,176,716324,No,Aalborg,business_owner,High
606,39,91,167,716742,No,Kolding,student,High


In [307]:
df['occupation'].unique()

array(['private_job', 'student', 'government_job', 'retired',
       'unemployed', 'freelancer', 'business_owner'], dtype=object)

In [308]:
df['insurance_premium_category'].unique()

array(['Medium', 'Low', 'High'], dtype=object)

In [309]:
df_feat = df.copy()

In [310]:
df_feat

,age,weight,height,income_dkk,smoker,city,occupation,insurance_premium_category
0,56,83,174,539178,No,Esbjerg,private_job,Medium
1,36,77,170,367498,No,Aalborg,student,Low
2,39,56,183,474027,No,Aarhus,government_job,Medium
3,38,87,171,627449,No,Silkeborg,retired,Medium
4,44,96,187,282747,No,Fredericia,unemployed,Low
...,...,...,...,...,...,...,...,...
995,63,82,177,449439,No,Aarhus,retired,Medium
996,59,73,176,441771,No,Copenhagen,private_job,Medium
997,20,97,185,693026,No,Aalborg,business_owner,High
998,24,90,175,505687,No,Fredericia,retired,Medium


In [311]:
# Feature 1: BMI
df_feat["bmi"] = df_feat["weight"] / ((df_feat["height"] / 100) ** 2)

In [312]:
# Feature 2: Age Group
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    return "senior"

In [313]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [314]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] == "Yes" and row["bmi"] >= 30:
        return "very_high"
    elif row["smoker"] == "Yes" and row["bmi"] >= 27:
        return "high"
    elif row["bmi"] >= 27:
        return "medium"
    else:
        return "low"

In [315]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1) #function is applied row by row

In [316]:
tier_1_cities = [
    "Copenhagen",
    "Aarhus",
    "Odense",
    "Aalborg"
]

tier_2_cities = [
    "Esbjerg",
    "Randers",
    "Kolding",
    "Horsens",
    "Vejle",
    "Roskilde",
    "Herning",
    "Silkeborg",
    "Næstved",
    "Fredericia",
    "Viborg"
]


In [317]:
# Feature 4: City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [318]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [319]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_dkk', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)


,income_dkk,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
729,550741,private_job,28.734694,senior,medium,2,Medium
415,377852,freelancer,21.200991,middle_aged,low,2,High
948,595783,retired,30.739417,adult,medium,1,Medium
969,304354,government_job,19.597842,young,low,1,Low
645,704032,retired,31.346939,middle_aged,medium,2,High


In [320]:

# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_dkk", "occupation"]]
y = df_feat["insurance_premium_category"]


In [321]:
X


,bmi,age_group,lifestyle_risk,city_tier,income_dkk,occupation
0,27.414454,middle_aged,medium,2,539178,private_job
1,26.643599,adult,low,1,367498,student
2,16.721909,adult,low,1,474027,government_job
3,29.752744,adult,medium,2,627449,retired
4,27.452887,adult,medium,2,282747,unemployed
...,...,...,...,...,...,...
995,26.173833,senior,low,1,449439,retired
996,23.566632,middle_aged,low,1,441771,private_job
997,28.341855,young,medium,1,693026,business_owner
998,29.387755,young,medium,2,505687,retired


In [322]:
# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)


In [323]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation"]
numeric_features = ["bmi", "income_dkk","city_tier"]

In [325]:
# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [326]:
model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

In [327]:
# Create a pipeline 
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", model)
])

In [328]:

pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_dkk',
                                                   'city_tier'])])),
                ('classifier',
                 GradientBoostingClassifier(learning_rate=0.05,
                                            n_estimators=300,
                                            random_state=42))])

In [329]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.855

In [330]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,income_dkk,occupation
255,23.939481,adult,low,2,437661,private_job
479,28.086420,adult,medium,2,448509,retired
368,16.516895,young,low,2,667061,unemployed
318,26.309431,middle_aged,low,1,768096,unemployed
484,25.978680,middle_aged,low,2,456806,business_owner


In [331]:
import os
import pickle

# Make sure folder exists
folder = '/content/Models'
os.makedirs(folder, exist_ok=True)

# Save the model
pickle_model_path = os.path.join(folder, "insurance_premium_predictor_model.pkl")
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)

print("Model saved at:", pickle_model_path)

Model saved at: /content/Models/insurance_premium_predictor_model.pkl
